# Pipeline 2 — Donor Upgrade Propensity (Next Ask Amount)

**Created:** 2026-04-06T23:33:27.172909Z

This notebook follows **CRISP-DM** while also satisfying the IS455 rubric sections:
- Problem Framing
- Data Acquisition, Preparation & Exploration
- Modeling & Feature Selection
- Evaluation & Interpretation
- Causal and Relationship Analysis
- Deployment Notes


## CRISP-DM Overview

### 1) Business Understanding
Goal: predict next donation amount to personalize fundraising asks and increase donation growth.

### 2) Data Understanding
Use monetary donations per supporter in sequence; predict the next donation amount within 180 days.

### 3) Data Preparation
Build per-donation training rows with history features; time-based split to avoid leakage.

### 4) Modeling
Predictive: Gradient Boosting Regressor. Explanatory: Linear Regression (log-scale).

### 5) Evaluation
Evaluate MAE/RMSE and discuss over/under-asking consequences for donors.

### 6) Deployment
Export predicted next amount and ask tier per supporter; import into `/api/ml/import`.


## 1) Problem Framing (Rubric)

State:
- the business question,
- who cares,
- why it matters,
- predictive vs explanatory goals.

We build **two models**:
- Predictive (optimize out-of-sample performance)
- Explanatory (interpretability / relationship analysis)


## 2) Data Acquisition, Preparation & Exploration (Rubric)

Rules to avoid leakage:
- Define an **as-of date** (cutoff).
- Build features using only data **on or before** the cutoff.
- Create labels using only data **after** the cutoff in a defined horizon.


In [ ]:
import json
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor


REPO_ROOT = Path("..").resolve()
RAW_DIR_A = (REPO_ROOT / "data" / "raw" / "lighthouse_csv_v7").resolve()
RAW_DIR_B = (REPO_ROOT / "data" / "raw").resolve()
DATA_DIR = RAW_DIR_A if RAW_DIR_A.exists() else RAW_DIR_B

OUT_DIR = (REPO_ROOT / "output" / "ml-predictions").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Out dir:", OUT_DIR)


def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}.")
    return pd.read_csv(path, encoding="utf-8-sig")


# After POST /api/admin/lighthouse-import, train from Azure SQL by setting INTEX_ODBC to your ODBC connection string.
SQL_TABLE_BY_STEM = {
    "supporters": "Supporters",
    "donations": "Contributions",
    "social_media_posts": "SocialMediaPosts",
    "safehouse_monthly_metrics": "SafehouseMonthlyMetrics",
    "residents": "Residents",
    "incident_reports": "IncidentReports",
    "home_visitations": "HomeVisitations",
    "process_recordings": "ProcessRecordings",
    "education_records": "EducationRecords",
    "health_wellbeing_records": "HealthWellbeingRecords",
}


def load_df(stem: str) -> pd.DataFrame:
    odbc = os.environ.get("INTEX_ODBC")
    table = SQL_TABLE_BY_STEM.get(stem)
    if odbc and table:
        try:
            import pyodbc

            with pyodbc.connect(odbc, timeout=120) as cnx:
                df = pd.read_sql(f"SELECT * FROM [{table}]", cnx)
            print(f"DB [{table}] rows:", len(df))
            if len(df) > 0:

                def pascal_to_snake(name: str) -> str:
                    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

                df = df.rename(columns={c: pascal_to_snake(str(c)) for c in df.columns})
                if stem == "donations":
                    df = df.rename(columns={"contribution_id": "donation_id"})
                if stem == "process_recordings":
                    df = df.rename(columns={"process_recording_id": "recording_id"})
                if stem == "home_visitations":
                    df = df.rename(columns={"home_visitation_id": "visitation_id"})
                return df
        except Exception as ex:
            print("INTEX_ODBC failed, using CSV:", ex)
    return require_csv(stem)


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.date


def to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(prediction_type: str, entity_type: str, df_out: pd.DataFrame, id_col: str, score_col: str, label_col: str | None = None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    rows = []
    for _, r in df_out.iterrows():
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(r[id_col]),
                "score": float(r[score_col]),
                "label": None if label_col is None else (None if pd.isna(r[label_col]) else str(r[label_col])),
                "payloadJson": json.dumps({k: v for k, v in r.items() if k not in {id_col, score_col, label_col}}, default=str),
            }
        )
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Wrote:", out_path, "rows=", len(rows))


In [ ]:
supporters = load_df("supporters")
        donations = load_df("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")

# Focus on monetary donations with numeric amount
mon = donations[(donations["donation_type"] == "Monetary") & donations["amount"].notna()].copy()
mon["amount"] = pd.to_numeric(mon["amount"], errors="coerce")
mon = mon.dropna(subset=["amount","donation_date"])

mon = mon.sort_values(["supporter_id","donation_date"]).copy()
mon["next_amount"] = mon.groupby("supporter_id")["amount"].shift(-1)
mon["next_date"] = mon.groupby("supporter_id")["donation_date"].shift(-1)
mon["days_to_next"] = (mon["next_date"] - mon["donation_date"]).dt.days

# Keep rows where a next donation exists within a reasonable horizon
df = mon[(mon["next_amount"].notna()) & (mon["days_to_next"].between(1, 180))].copy()

# Add supporter attributes
df = df.merge(
    supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]],
    on="supporter_id",
    how="left"
)

# Simple history features as of current donation
df["donations_so_far"] = df.groupby("supporter_id").cumcount() + 1
df["prev_amount"] = df.groupby("supporter_id")["amount"].shift(1)
df["prev_date"] = df.groupby("supporter_id")["donation_date"].shift(1)
df["recency_days"] = (df["donation_date"] - df["prev_date"]).dt.days
df["recency_days"] = df["recency_days"].fillna(9999).clip(lower=0)
df["prev_amount"] = df["prev_amount"].fillna(df["amount"])

# Target
df["y_next_amount"] = df["next_amount"]
df["y_upgrade_25pct"] = (df["next_amount"] >= (df["amount"] * 1.25)).astype(int)

print("Rows:", len(df))
df[["supporter_id","donation_date","amount","next_amount","days_to_next","y_upgrade_25pct"]].head()


In [ ]:
# Time-based train/test split (avoid leakage across time)
cutoff_date = df["donation_date"].max() - pd.Timedelta(days=90)
train = df[df["donation_date"] <= cutoff_date].copy()
test = df[df["donation_date"] > cutoff_date].copy()
print("Train rows:", len(train), "Test rows:", len(test), "Cutoff:", cutoff_date.date())

features = [
    "supporter_type","relationship_type","region","country","acquisition_channel",
    "amount","prev_amount","recency_days","donations_so_far","is_recurring","campaign_name","channel_source"
]

cat_cols = ["supporter_type","relationship_type","region","country","acquisition_channel","campaign_name","channel_source"]
num_cols = [c for c in features if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train, y_train = train[features], train["y_next_amount"]
X_test, y_test = test[features], test["y_next_amount"]


In [ ]:
# Predictive model (regression)
gbr = Pipeline(steps=[
    ("pre", pre),
    ("model", GradientBoostingRegressor(random_state=42))
])
gbr.fit(X_train, y_train)
pred = gbr.predict(X_test)
print("Predictive (GBReg):", eval_regression(y_test, pred))

# Explanatory model (linear regression on log(amount))
lin = Pipeline(steps=[
    ("pre", pre),
    ("model", LinearRegression())
])
lin.fit(X_train, np.log1p(y_train))
pred2 = np.expm1(lin.predict(X_test))
print("Explanatory (Linear):", eval_regression(y_test, pred2))


In [ ]:
# Score each supporter using their latest monetary donation (recommend next ask)
latest = mon.sort_values(["supporter_id","donation_date"]).groupby("supporter_id").tail(1).copy()
latest = latest.merge(
    supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]],
    on="supporter_id",
    how="left"
)
latest["donations_so_far"] = mon.groupby("supporter_id").size().reindex(latest["supporter_id"]).values
latest["prev_amount"] = mon.sort_values(["supporter_id","donation_date"]).groupby("supporter_id")["amount"].nth(-2).reindex(latest["supporter_id"]).values
latest["prev_amount"] = pd.to_numeric(latest["prev_amount"], errors="coerce").fillna(latest["amount"])

latest["prev_date"] = mon.sort_values(["supporter_id","donation_date"]).groupby("supporter_id")["donation_date"].nth(-2).reindex(latest["supporter_id"]).values
latest["recency_days"] = (latest["donation_date"] - latest["prev_date"]).dt.days
latest["recency_days"] = latest["recency_days"].fillna(9999).clip(lower=0)

for c in ["is_recurring","campaign_name","channel_source"]:
    if c not in latest.columns:
        latest[c] = None

latest["predicted_next_amount"] = gbr.predict(latest[features])
latest["upgrade_ratio"] = (latest["predicted_next_amount"] / latest["amount"]).replace([np.inf,-np.inf], np.nan).fillna(0)
latest["ask_tier"] = pd.cut(latest["upgrade_ratio"], bins=[-np.inf,1.05,1.25,1.5,np.inf], labels=["Maintain","Small Upgrade","Upgrade","Major Upgrade"])

export_predictions_json(
    prediction_type="donor_upgrade_next_amount",
    entity_type="Supporter",
    df_out=latest[["supporter_id","predicted_next_amount","ask_tier","amount","upgrade_ratio","recency_days","donations_so_far","acquisition_channel","supporter_type"]].rename(columns={"predicted_next_amount":"score"}),
    id_col="supporter_id",
    score_col="score",
    label_col="ask_tier"
)


## 3) Modeling & Feature Selection (Rubric)

- Predictive model: tree/ensemble
- Explanatory model: linear/logistic regression


## 4) Evaluation & Interpretation (Rubric)

Interpret in business terms, and discuss real-world costs of errors.

## 5) Causal and Relationship Analysis (Rubric)

Discuss relationships, confounding risks, and where correlation ≠ causation.


## 6) Deployment Notes (Rubric)

Export predictions to JSON and import into the deployed app:
- `POST /api/ml/import?replace=true` (admin-only)
- View in `/app/ml` (Staff Portal → ML Insights)
